# NB0 - Setup, Data Preparation, Smoke Test

Prepares all three datasets (Container v3, ICDAR 2015, COCO val2017), draws
box-overlay sanity images, smoke-tests YOLOv8s / RT-DETR-l / YOLO-World, and
pushes `ausdm-v3-data` for the four downstream notebooks. Runs top to bottom
from a clean kernel and skips completed stages on re-run.

**Kaggle setup required before running:** phone-verified account, internet toggled on,
Roboflow API key stored as Secret `ROBOFLOW_API_KEY`, dataset `icdar2015-task41-raw`
mounted at `/kaggle/input/icdar2015-task41-raw`.


## 0. One-time environment setup

In [ ]:
# One-time environment setup. Runs every session; -q keeps output small.
# Uninstall ray to avoid ultralytics 8.3.40's broken raytune callback on Kaggle.
! pip install -q ultralytics==8.3.40 roboflow==1.1.50 pillow==10.4.0
! pip uninstall -y -q ray 2>/dev/null || true


## 1. Imports, constants, seeds

In [ ]:
# Standard library
import json, os, sys, shutil, subprocess, time, random, re, zipfile, io, urllib.request
from pathlib import Path
from collections import Counter, defaultdict

# Block ultralytics 8.3.40's broken raytune callback: pretend `ray` is not installed
sys.modules['ray'] = None

# Numeric and plotting
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

# Deep learning
import torch

# Reproducibility
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths (Kaggle)
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')
DATA_ROOT      = KAGGLE_WORKING / 'data'
CONTAINER_DIR  = DATA_ROOT / 'container'
ICDAR_DIR      = DATA_ROOT / 'icdar2015'
COCO_DIR       = DATA_ROOT / 'coco'
SANITY_DIR     = KAGGLE_WORKING / 'sanity_images'
RUNS_DIR       = KAGGLE_WORKING / 'runs_smoke'
STATE_FILE     = KAGGLE_WORKING / 'nb0_state.json'

# Split ratios (plan section 5.1)
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.20
TEST_FRAC  = 0.10

# Roboflow project pin (plan section 8a)
RF_WORKSPACE = 'dissertation-t3buk'
RF_PROJECT   = 'container-number-detection'
RF_VERSION   = 3

for d in [DATA_ROOT, CONTAINER_DIR, ICDAR_DIR, COCO_DIR, SANITY_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'KAGGLE_WORKING = {KAGGLE_WORKING} (exists={KAGGLE_WORKING.exists()})')
print(f'DATA_ROOT      = {DATA_ROOT}')
print(f'SEED           = {SEED}')


## 2. Environment verification and version pinning

In [ ]:
# _record_versions - collect library versions for the paper's config table
def _record_versions():
    import ultralytics, PIL
    versions = {
        'python'     : sys.version.split()[0],
        'torch'      : torch.__version__,
        'cuda_avail' : torch.cuda.is_available(),
        'cuda'       : torch.version.cuda if torch.cuda.is_available() else None,
        'gpu'        : torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'ultralytics': ultralytics.__version__,
        'PIL'        : PIL.__version__,
        'numpy'      : np.__version__,
    }
    return versions

VERSIONS = _record_versions()
for k, v in VERSIONS.items():
    print(f'  {k:12s} = {v}')

# Persist for the paper
(KAGGLE_WORKING / 'versions.json').write_text(json.dumps(VERSIONS, indent=2))


## 3. Secret assertion - fail before any compute

In [ ]:
# _get_roboflow_key - fetch Kaggle Secret, fail loudly if missing
def _get_roboflow_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    except Exception as e:
        raise RuntimeError(
            'ROBOFLOW_API_KEY Kaggle Secret not found. Add-ons -> Secrets -> '
            'Add a new secret named ROBOFLOW_API_KEY, attach to this notebook, '
            'then restart the kernel. Original error: %s' % e
        )

ROBOFLOW_API_KEY = _get_roboflow_key()
assert len(ROBOFLOW_API_KEY) > 10, 'Roboflow key looks too short'

# ICDAR raw dataset mount check - auto-detect the slug under /kaggle/input
# (Kaggle sometimes appends a suffix if the slug collides; scan for the marker file/folder)
def _find_icdar_mount():
    # Walk /kaggle/input up to 3 levels deep looking for a directory that
    # contains the training-images marker (as folder or zip).
    if not KAGGLE_INPUT.exists():
        return None
    def _has_marker(d):
        return ((d / 'ch4_training_images').exists() or
                (d / 'ch4_training_images.zip').exists())
    # Prefer the exact slug
    if (KAGGLE_INPUT / 'icdar2015-task41-raw').exists() and _has_marker(KAGGLE_INPUT / 'icdar2015-task41-raw'):
        return KAGGLE_INPUT / 'icdar2015-task41-raw'
    # Walk
    for root, dirs, _files in os.walk(KAGGLE_INPUT):
        depth = Path(root).relative_to(KAGGLE_INPUT).parts
        if len(depth) > 3:
            dirs[:] = []
            continue
        rp = Path(root)
        if _has_marker(rp):
            return rp
    return None

ICDAR_MOUNT = _find_icdar_mount()
assert ICDAR_MOUNT is not None, (
    'ICDAR raw dataset not mounted. Right sidebar -> Add Data -> search '
    '"icdar2015-task41-raw" -> Add, THEN re-run this cell. '
    f'Currently /kaggle/input has: {[p.name for p in KAGGLE_INPUT.iterdir()] if KAGGLE_INPUT.exists() else "nothing"}'
)
print(f'Roboflow key: OK ({len(ROBOFLOW_API_KEY)} chars)')
print(f'ICDAR mount : OK ({ICDAR_MOUNT})')
print(f'ICDAR files : {sorted(p.name for p in ICDAR_MOUNT.iterdir())}')


## 4. Resumable state (skip completed stages on re-run)

In [ ]:
# _load_state / _mark_done - stage-level idempotency for interrupted sessions
def _load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {}

def _mark_done(stage, extra=None):
    st = _load_state()
    st[stage] = {'done': True, 'ts': time.time()}
    if extra:
        st[stage].update(extra)
    STATE_FILE.write_text(json.dumps(st, indent=2))

def _is_done(stage):
    return _load_state().get(stage, {}).get('done', False)

print('current state:', json.dumps(_load_state(), indent=2))


## 5. Container v3 - download and re-split with seed 0

Roboflow's own split is already 70/20/10 and clean (verified Day 0), but we
re-split from a pooled 1001-image list with `seed 0` so the paper's split
comes from our code rather than Roboflow's export.

In [ ]:
# _download_container - Roboflow SDK, pinned to v3
def _download_container():
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    proj = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    tgt  = KAGGLE_WORKING / f'rf_raw_v{RF_VERSION}'
    if not tgt.exists():
        ver = proj.version(RF_VERSION)
        ver.download('yolov8', location=str(tgt))
    return tgt

if not _is_done('container_download'):
    RF_TGT = _download_container()
    print('Roboflow download OK:', RF_TGT)
    _mark_done('container_download', {'path': str(RF_TGT)})
else:
    RF_TGT = Path(_load_state()['container_download']['path'])
    print('SKIP container_download, existing at', RF_TGT)

# Sanity: expect 1001 total across the three sub-splits
total = sum(len(list((RF_TGT / s / 'images').iterdir())) for s in ['train','valid','test'])
assert total == 1001, f'Expected 1001 images, got {total}'
print(f'Total images downloaded: {total}')


In [ ]:
# _resplit_container - pool all 1001, shuffle with seed 0, re-partition 700/200/101
def _resplit_container(rf_tgt, out_dir):
    if out_dir.exists() and any(out_dir.iterdir()):
        shutil.rmtree(out_dir)
    for split in ['train','valid','test']:
        for sub in ['images','labels']:
            (out_dir / split / sub).mkdir(parents=True, exist_ok=True)

    # Pool
    items = []  # (img_path, lab_path)
    for split in ['train','valid','test']:
        for img in sorted((rf_tgt / split / 'images').iterdir()):
            lab = rf_tgt / split / 'labels' / (img.stem + '.txt')
            items.append((img, lab))
    items.sort(key=lambda t: t[0].name)  # deterministic before shuffle

    rng = random.Random(SEED)
    rng.shuffle(items)

    n = len(items)
    n_train = int(round(n * TRAIN_FRAC))
    n_val   = int(round(n * VAL_FRAC))
    n_test  = n - n_train - n_val
    splits_plan = {
        'train': items[:n_train],
        'valid': items[n_train:n_train+n_val],
        'test' : items[n_train+n_val:],
    }
    for split, pairs in splits_plan.items():
        for img, lab in pairs:
            shutil.copy2(img, out_dir / split / 'images' / img.name)
            if lab.exists():
                shutil.copy2(lab, out_dir / split / 'labels' / lab.name)
    return {s: len(v) for s, v in splits_plan.items()}

if not _is_done('container_resplit'):
    counts = _resplit_container(RF_TGT, CONTAINER_DIR)
    print('re-split counts:', counts)
    _mark_done('container_resplit', {'counts': counts})
else:
    counts = _load_state()['container_resplit']['counts']
    print('SKIP container_resplit, counts:', counts)

# Write dataset YAML
container_yaml = CONTAINER_DIR / 'data.yaml'
container_yaml.write_text(f'''path: {CONTAINER_DIR}
train: train/images
val:   valid/images
test:  test/images
nc: 1
names: ['container-number']
''')
print('container data.yaml:', container_yaml)


## 6. ICDAR 2015 - unpack and polygon-to-axis-aligned conversion

Four zips arrive from rrc.cvc.uab.es Challenge 4. Training GT: `x1,y1,x2,y2,x3,y3,x4,y4,transcription`;
test GT (`Challenge4_Test_Task1_GT.zip`) has only the 8 coordinate columns. Transcription `###` means
illegible / don't-care and is dropped. Boxes are the tight axis-aligned box
around the quad.

In [ ]:
# _unpack_icdar - handle either raw zips or Kaggle-auto-extracted folders
# Kaggle Datasets auto-extract .zip files on upload by default, so the mount may
# already contain the four extracted folders instead of the zips. Detect either.
TEST_GT_CAND_ZIP = ['Challenge4_Test_Task1_GT.zip',
                    'Challenge4_Test_Task4_GT.zip',
                    'ch4_test_localization_transcription_gt.zip',
                    'ch4_test_gt.zip']
TEST_GT_CAND_DIR = ['Challenge4_Test_Task1_GT',
                    'Challenge4_Test_Task4_GT',
                    'ch4_test_localization_transcription_gt',
                    'ch4_test_gt']

def _unpack_icdar():
    zips = {
        'train_images': 'ch4_training_images.zip',
        'train_gt'    : 'ch4_training_localization_transcription_gt.zip',
        'test_images' : 'ch4_test_images.zip',
        'test_gt'     : None,
    }
    for cand in TEST_GT_CAND_ZIP:
        if (ICDAR_MOUNT / cand).exists():
            zips['test_gt'] = cand; break

    # If none of the zips exist, assume Kaggle auto-extracted them into folders
    any_zip = any((ICDAR_MOUNT / v).exists() for v in zips.values() if v)
    if not any_zip:
        dirs = {
            'train_images': 'ch4_training_images',
            'train_gt'    : 'ch4_training_localization_transcription_gt',
            'test_images' : 'ch4_test_images',
            'test_gt'     : None,
        }
        for cand in TEST_GT_CAND_DIR:
            if (ICDAR_MOUNT / cand).is_dir():
                dirs['test_gt'] = cand; break
        assert dirs['test_gt'], 'No test GT folder or zip found in mount'
        for k, d in dirs.items():
            assert (ICDAR_MOUNT / d).is_dir(), f'Missing folder: {d}'
        return ICDAR_MOUNT, dirs

    assert zips['test_gt'], 'Test GT zip not found - check upload naming'
    stage_dir = KAGGLE_WORKING / 'icdar_raw'
    stage_dir.mkdir(exist_ok=True)
    for key, name in zips.items():
        out = stage_dir / key
        if out.exists() and any(out.iterdir()):
            continue
        out.mkdir(exist_ok=True)
        with zipfile.ZipFile(ICDAR_MOUNT / name) as z:
            z.extractall(out)
    dirs = {k: k for k in zips}
    return stage_dir, dirs

if not _is_done('icdar_unpack'):
    ICDAR_RAW, ICDAR_SUBDIRS = _unpack_icdar()
    print('ICDAR raw root:', ICDAR_RAW)
    print('subdirs       :', ICDAR_SUBDIRS)
    _mark_done('icdar_unpack', {'path': str(ICDAR_RAW), 'subdirs': ICDAR_SUBDIRS})
else:
    st = _load_state()['icdar_unpack']
    ICDAR_RAW = Path(st['path'])
    ICDAR_SUBDIRS = st['subdirs']
    print('SKIP icdar_unpack, root =', ICDAR_RAW)


In [ ]:
# _icdar_convert - polygons -> axis-aligned YOLO labels; drop '###' transcriptions
# The ICDAR archives sometimes lack a subfolder; auto-locate the images/gt roots.

def _find_dir_with(root, glob_pat):
    if any(root.glob(glob_pat)):
        return root
    for p in root.rglob('*'):
        if p.is_dir() and any(p.glob(glob_pat)):
            return p
    return root

def _convert_pair(img_dir, gt_dir, out_split):
    (ICDAR_DIR / out_split / 'images').mkdir(parents=True, exist_ok=True)
    (ICDAR_DIR / out_split / 'labels').mkdir(parents=True, exist_ok=True)
    n_img = n_box = n_drop = 0
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
        gt = gt_dir / f'gt_{img.stem}.txt'
        if not gt.exists(): continue
        text = gt.read_text(encoding='utf-8-sig', errors='ignore')
        im = Image.open(img); W, H = im.size
        lines_out = []
        for ln in text.splitlines():
            if not ln.strip(): continue
            parts = ln.split(',', 8)
            if len(parts) < 8: continue
            try:
                xs = [float(parts[i]) for i in (0,2,4,6)]
                ys = [float(parts[i]) for i in (1,3,5,7)]
            except ValueError: continue
            trans = parts[8].strip() if len(parts) >= 9 else ''
            if trans == '###':
                n_drop += 1; continue
            x_min, x_max = max(0, min(xs)), min(W, max(xs))
            y_min, y_max = max(0, min(ys)), min(H, max(ys))
            bw = x_max - x_min; bh = y_max - y_min
            if bw <= 1 or bh <= 1: continue
            cx = (x_min + x_max)/2/W; cy = (y_min + y_max)/2/H
            lines_out.append(f'0 {cx:.6f} {cy:.6f} {bw/W:.6f} {bh/H:.6f}')
            n_box += 1
        (ICDAR_DIR / out_split / 'labels' / f'{img.stem}.txt').write_text(
            '\n'.join(lines_out) + ('\n' if lines_out else '')
        )
        shutil.copy2(img, ICDAR_DIR / out_split / 'images' / img.name)
        n_img += 1
    return n_img, n_box, n_drop

if not _is_done('icdar_convert'):
    train_img_dir = _find_dir_with(ICDAR_RAW / ICDAR_SUBDIRS['train_images'], '*.jpg')
    train_gt_dir  = _find_dir_with(ICDAR_RAW / ICDAR_SUBDIRS['train_gt'],     'gt_*.txt')
    test_img_dir  = _find_dir_with(ICDAR_RAW / ICDAR_SUBDIRS['test_images'],  '*.jpg')
    test_gt_dir   = _find_dir_with(ICDAR_RAW / ICDAR_SUBDIRS['test_gt'],      'gt_*.txt')
    print('resolved dirs:')
    print(' train img:', train_img_dir)
    print(' train gt :', train_gt_dir)
    print(' test  img:', test_img_dir)
    print(' test  gt :', test_gt_dir)

    tr = _convert_pair(train_img_dir, train_gt_dir, 'train')
    te = _convert_pair(test_img_dir,  test_gt_dir,  'test')
    print(f'train: {tr[0]} imgs, {tr[1]} boxes, {tr[2]} dropped (###)')
    print(f'test : {te[0]} imgs, {te[1]} boxes, {te[2]} dropped (###)')
    assert tr[0] >= 900, f'ICDAR train expected ~1000, got {tr[0]}'
    assert te[0] >= 450, f'ICDAR test expected ~500, got {te[0]}'
    _mark_done('icdar_convert', {'train_imgs': tr[0], 'test_imgs': te[0],
                                 'train_boxes': tr[1], 'test_boxes': te[1]})
else:
    print('SKIP icdar_convert:', _load_state()['icdar_convert'])

icdar_yaml = ICDAR_DIR / 'data.yaml'
icdar_yaml.write_text(f'''path: {ICDAR_DIR}
train: train/images
val:   test/images
test:  test/images
nc: 1
names: ['text']
''')
print('icdar data.yaml:', icdar_yaml)


## 7. COCO val2017 - val-only YAML

The default ultralytics `coco.yaml` pulls the 19 GB train2017 archive. We write
a YAML whose `train` and `val` both point at val2017, so `model.val()` runs
only against the 5000-image validation set.

In [ ]:
# _fetch_coco_val - images + YOLO-format labels
COCO_IMAGES_ZIP = 'http://images.cocodataset.org/zips/val2017.zip'
COCO_LABELS_ZIP = 'https://github.com/ultralytics/yolov5/releases/download/v1.0/coco2017labels.zip'

def _fetch_coco_val():
    val_img_dir = COCO_DIR / 'images' / 'val2017'
    val_lab_dir = COCO_DIR / 'labels' / 'val2017'
    if val_img_dir.exists() and len(list(val_img_dir.glob('*.jpg'))) >= 4900 and val_lab_dir.exists():
        return val_img_dir, val_lab_dir
    tmp = KAGGLE_WORKING / 'coco_download'
    tmp.mkdir(exist_ok=True)

    imgs_zip = tmp / 'val2017.zip'
    labs_zip = tmp / 'coco2017labels.zip'
    if not imgs_zip.exists():
        print('downloading val2017 (~780 MB)...')
        urllib.request.urlretrieve(COCO_IMAGES_ZIP, imgs_zip)
    if not labs_zip.exists():
        print('downloading YOLO labels (~46 MB)...')
        urllib.request.urlretrieve(COCO_LABELS_ZIP, labs_zip)

    (COCO_DIR / 'images').mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(imgs_zip) as z:
        z.extractall(COCO_DIR / 'images')

    labs_stage = tmp / 'labels_stage'
    labs_stage.mkdir(exist_ok=True)
    with zipfile.ZipFile(labs_zip) as z:
        z.extractall(labs_stage)

    # Layout inside coco2017labels.zip: coco/labels/val2017/*.txt
    src = None
    for cand in labs_stage.rglob('val2017'):
        if cand.is_dir() and any(cand.glob('*.txt')):
            src = cand; break
    assert src is not None, 'Could not find labels/val2017 in the labels zip'
    val_lab_dir.parent.mkdir(parents=True, exist_ok=True)
    if val_lab_dir.exists():
        shutil.rmtree(val_lab_dir)
    shutil.move(str(src), str(val_lab_dir))
    return val_img_dir, val_lab_dir

if not _is_done('coco_fetch'):
    IMG, LAB = _fetch_coco_val()
    n_img = len(list(IMG.glob('*.jpg'))); n_lab = len(list(LAB.glob('*.txt')))
    print(f'val2017 images: {n_img}, labels: {n_lab}')
    assert n_img >= 4900, f'Expected 5000, got {n_img}'
    _mark_done('coco_fetch', {'images': n_img, 'labels': n_lab})
else:
    print('SKIP coco_fetch:', _load_state()['coco_fetch'])


In [ ]:
# _write_coco_yaml - both train and val point at val2017 to block the 19 GB pull
COCO_NAMES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light',
    'fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow',
    'elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee',
    'skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle',
    'wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich','orange',
    'broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant','bed',
    'dining table','toilet','tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear','hair drier','toothbrush',
]
coco_yaml = COCO_DIR / 'coco_val_only.yaml'
coco_yaml.write_text(f'''path: {COCO_DIR}
train: images/val2017
val:   images/val2017
test:  images/val2017
nc: 80
names: {COCO_NAMES}
''')
print('coco YAML written to', coco_yaml)


## 8. Dataset characterisation table

In [ ]:
# _characterise - single-row-per-split summary for the paper's dataset table
def _characterise(name, root, splits):
    rows = []
    for s in splits:
        img_dir = root / s / 'images'
        lab_dir = root / s / 'labels'
        if not img_dir.exists():
            continue
        imgs = sorted(img_dir.iterdir())
        n_img = len(imgs)
        n_box = 0; box_areas = []; sizes = Counter()
        for img in imgs:
            lab = lab_dir / f'{img.stem}.txt'
            if lab.exists():
                for ln in lab.read_text().splitlines():
                    p = ln.strip().split()
                    if len(p) >= 5:
                        n_box += 1
                        box_areas.append(float(p[3]) * float(p[4]))
            with Image.open(img) as im:
                sizes[im.size] += 1
        size_str = ', '.join(f'{w}x{h}:{c}' for (w,h), c in sizes.most_common(3))
        rows.append({
            'dataset': name, 'split': s, 'images': n_img,
            'boxes': n_box,
            'boxes_per_img': round(n_box / max(n_img,1), 2),
            'box_area_p50': round(float(np.median(box_areas)), 5) if box_areas else None,
            'box_area_p95': round(float(np.percentile(box_areas, 95)), 5) if box_areas else None,
            'sizes_top3': size_str,
        })
    return rows

rows = []
rows += _characterise('container', CONTAINER_DIR, ['train','valid','test'])
rows += _characterise('icdar',     ICDAR_DIR,     ['train','test'])
# COCO is pretrained-eval-only; still count it for the table
coco_rows = []
coco_val_imgs = COCO_DIR / 'images' / 'val2017'
if coco_val_imgs.exists():
    n = len(list(coco_val_imgs.glob('*.jpg')))
    coco_rows.append({'dataset': 'coco', 'split': 'val2017', 'images': n,
                      'boxes': None, 'boxes_per_img': None,
                      'box_area_p50': None, 'box_area_p95': None,
                      'sizes_top3': 'variable'})
rows += coco_rows

df_stats = pd.DataFrame(rows)
df_stats.to_csv(KAGGLE_WORKING / 'dataset_characterisation.csv', index=False)
print(df_stats.to_string(index=False))


## 9. Sanity image render (6 per split, with boxes)

**Mandatory. Reviewer #2 caught v2 Figure 3 showing raw images without boxes.**
Every downstream figure gets the same visual check.

In [ ]:
# _render_sanity - draw label boxes on 6 samples per split; verify visually before moving on
def _render_sanity(name, root, splits, n_per=6):
    rng = random.Random(SEED)
    for s in splits:
        img_dir = root / s / 'images'; lab_dir = root / s / 'labels'
        if not img_dir.exists(): continue
        names = [p.name for p in img_dir.iterdir()]
        rng.shuffle(names)
        chosen = []
        for n in names:
            lab = lab_dir / f'{Path(n).stem}.txt'
            if lab.exists() and lab.stat().st_size > 0:
                chosen.append(n)
            if len(chosen) == n_per: break
        for n in chosen:
            im = Image.open(img_dir / n).convert('RGB'); W, H = im.size
            dr = ImageDraw.Draw(im)
            with open(lab_dir / f'{Path(n).stem}.txt') as f:
                for ln in f:
                    p = ln.strip().split()
                    if len(p) < 5: continue
                    cx, cy, bw, bh = map(float, p[1:5])
                    x1 = (cx-bw/2)*W; y1 = (cy-bh/2)*H
                    x2 = (cx+bw/2)*W; y2 = (cy+bh/2)*H
                    dr.rectangle([x1,y1,x2,y2], outline='lime', width=4)
            im.save(SANITY_DIR / f'{name}_{s}_{Path(n).stem}.png')

_render_sanity('container', CONTAINER_DIR, ['train','valid','test'])
_render_sanity('icdar',     ICDAR_DIR,     ['train','test'])
print('sanity images written to', SANITY_DIR)
print(sorted(p.name for p in SANITY_DIR.iterdir())[:6], '...')


**9.1 Explanation.** Every red box below must sit on a container number
(container dataset) or a text region (ICDAR). An image with no visible box, or
a box in the wrong place, is the same defect Reviewer #2 flagged in v2 Figure 3.

In [ ]:
# Inline preview of 3 container sanity images so the visual check happens in the notebook
from IPython.display import Image as IPImg, display
for name in sorted(SANITY_DIR.glob('container_*'))[:3]:
    print(name.name); display(IPImg(str(name), width=400))


## 10. Smoke tests - 10 images, 1 epoch, all three model families

Fails-here, not-fails-on-Day-1 principle. The YOLO-World cell is the fragile
one; if it dies, decide between swapping the variant and dropping the
open-vocabulary arm before spending compute on the supervised arm.

In [ ]:
# _make_smoke_subset - 10 image / label pairs into a temp dataset
def _make_smoke_subset(src_dir, dst_dir, n=10):
    if dst_dir.exists(): shutil.rmtree(dst_dir)
    (dst_dir / 'train' / 'images').mkdir(parents=True, exist_ok=True)
    (dst_dir / 'train' / 'labels').mkdir(parents=True, exist_ok=True)
    (dst_dir / 'valid' / 'images').mkdir(parents=True, exist_ok=True)
    (dst_dir / 'valid' / 'labels').mkdir(parents=True, exist_ok=True)
    imgs = sorted((src_dir / 'train' / 'images').iterdir())[:n]
    for i, img in enumerate(imgs):
        lab = src_dir / 'train' / 'labels' / f'{img.stem}.txt'
        split = 'train' if i < 8 else 'valid'
        shutil.copy2(img, dst_dir / split / 'images' / img.name)
        if lab.exists():
            shutil.copy2(lab, dst_dir / split / 'labels' / lab.name)
    (dst_dir / 'data.yaml').write_text(f'''path: {dst_dir}
train: train/images
val:   valid/images
nc: 1
names: ['container-number']
''')
    return dst_dir / 'data.yaml'

SMOKE_YAML = _make_smoke_subset(CONTAINER_DIR, KAGGLE_WORKING / 'smoke_data')
print('smoke yaml:', SMOKE_YAML)


In [ ]:
# Neutralise the raytune callback FUNCTION so any cached reference is a no-op.
# ray 3.x removed `ray.train._internal.session._get_session` which the ultralytics 8.3.40
# raytune callback still calls. Rewriting the function's __code__ makes every existing
# reference to it (including whatever the trainer already registered) a silent pass.
import sys
try:
    from ultralytics.utils.callbacks import raytune as _rt
    _rt.on_fit_epoch_end.__code__ = (lambda trainer: None).__code__
    _rt.callbacks = {}
    _rt.tune = None
    print('raytune callback neutralised (code swapped to no-op)')
except Exception as e:
    print('raytune patch skipped:', e)


In [ ]:
# Smoke YOLOv8s
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
r = model.train(data=str(SMOKE_YAML), epochs=1, imgsz=640, batch=4,
                project=str(RUNS_DIR), name='smoke_yolov8s',
                verbose=False, save=False, plots=False, exist_ok=True)
print('YOLOv8s smoke: OK')


In [ ]:
# Smoke RT-DETR-l  (batch 4 for the 10-image smoke; NB1 uses batch 8 on the full set)
from ultralytics import RTDETR
model = RTDETR('rtdetr-l.pt')
r = model.train(data=str(SMOKE_YAML), epochs=1, imgsz=640, batch=4,
                project=str(RUNS_DIR), name='smoke_rtdetr',
                verbose=False, save=False, plots=False, exist_ok=True)
print('RT-DETR-l smoke: OK')


In [ ]:
# Smoke YOLO-World fine-tune  (single most-likely failure - plan section 9)
from ultralytics import YOLOWorld
try:
    model = YOLOWorld('yolov8s-worldv2.pt')
    model.set_classes(['container-number'])
    r = model.train(data=str(SMOKE_YAML), epochs=1, imgsz=640, batch=4,
                    project=str(RUNS_DIR), name='smoke_yoloworld',
                    verbose=False, save=False, plots=False, exist_ok=True)
    print('YOLO-World smoke: OK')
except Exception as e:
    print('YOLO-World SMOKE FAILED:', repr(e))
    print('Decide tonight: swap variant or drop the open-vocab arm.')
    raise


**10.1 Explanation.** All three prints must say `OK`. A YOLO-World
failure is expected to be library-version-related - most fixes are
`pip install ultralytics==<version>` in cell 0. If it cannot be fixed
tonight, NB2 must be dropped, which changes the paper's central claim.

## 11. Authenticated dry-run push

Fail on permission / naming problems now, not after 10 hours of compute
tomorrow. `kaggle datasets create` writes the initial version; later notebooks
push updates with `kaggle datasets version`.

In [ ]:
# _write_metadata - Kaggle dataset descriptor
DATASET_SLUG = 'ausdm-v3-data'          # if this clashes with an existing dataset in your account, rename here
def _kaggle_user():
    try:
        return json.loads((Path.home() / '.kaggle' / 'kaggle.json').read_text())['username']
    except Exception:
        return os.environ.get('KAGGLE_USERNAME', 'unknown')

KAGGLE_USER = _kaggle_user()
print('Kaggle username:', KAGGLE_USER)

PUSH_DIR = KAGGLE_WORKING / 'push_stage'
if PUSH_DIR.exists(): shutil.rmtree(PUSH_DIR)
PUSH_DIR.mkdir(exist_ok=True)

# Minimal payload for the dry-run: the yamls, stats CSV, version info, sanity previews
shutil.copy2(container_yaml,        PUSH_DIR / 'container_data.yaml')
shutil.copy2(icdar_yaml,            PUSH_DIR / 'icdar_data.yaml')
shutil.copy2(coco_yaml,             PUSH_DIR / 'coco_val_only.yaml')
shutil.copy2(KAGGLE_WORKING / 'versions.json',                PUSH_DIR / 'versions.json')
shutil.copy2(KAGGLE_WORKING / 'dataset_characterisation.csv', PUSH_DIR / 'dataset_characterisation.csv')
(PUSH_DIR / 'sanity').mkdir(exist_ok=True)
for p in list(SANITY_DIR.iterdir())[:12]:
    shutil.copy2(p, PUSH_DIR / 'sanity' / p.name)

metadata = {
    'title': 'AusDM v3 shared data',
    'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC-BY-SA-4.0'}]
}
(PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2))
print('push staged at', PUSH_DIR)
print('metadata:', json.dumps(metadata, indent=2))


In [ ]:
# Dry-run: try create, fall back to version bump if the dataset already exists
def _push():
    r = subprocess.run(['kaggle','datasets','create','-p',str(PUSH_DIR),'-r','zip'],
                       capture_output=True, text=True)
    print('CREATE stdout:', r.stdout.strip())
    print('CREATE stderr:', r.stderr.strip())
    if r.returncode != 0 and 'already exists' in (r.stderr + r.stdout).lower():
        r = subprocess.run(['kaggle','datasets','version','-p',str(PUSH_DIR),
                            '-m','NB0 dry-run','-r','zip'],
                           capture_output=True, text=True)
        print('VERSION stdout:', r.stdout.strip())
        print('VERSION stderr:', r.stderr.strip())
    return r.returncode

rc = _push()
if rc != 0:
    print('WARNING: push failed. Fix authentication before running NB1..NB4.')
else:
    print('push OK')
    _mark_done('dry_run_push')


## 12. Handoff to NB1 / NB2 / NB3 / NB4

At this point `ausdm-v3-data` is on Kaggle with:

- `container_data.yaml`, `icdar_data.yaml`, `coco_val_only.yaml`
- `dataset_characterisation.csv`
- `versions.json` (for the paper's config table)
- `sanity/` - 12 preview images with boxes drawn

**Do not proceed to NB1 unless section 10 printed OK three times and every
inline sanity image has a red box sitting on the correct region.**


In [ ]:
print('NB0 state:', json.dumps(_load_state(), indent=2))
print()
print('=== Cheat sheet ===')
print(f'container YAML : {container_yaml}')
print(f'icdar YAML     : {icdar_yaml}')
print(f'coco YAML      : {coco_yaml}')
print(f'stats CSV      : {KAGGLE_WORKING / "dataset_characterisation.csv"}')
print(f'sanity dir     : {SANITY_DIR}')
